# Advanced Problems with Solutions: Partial Functions

This notebook contains advanced practice problems for `functools.partial` in Python, including argument binding, keyword overriding, introspection, mutable state, callbacks, sorting keys, decorators, and practical design patterns.

In [1]:
from functools import partial, update_wrapper
import inspect
from operator import mul, itemgetter
from datetime import datetime

## Problem 1: Predict Argument Binding

Given the function below, create a partial function that fixes `a=10` and `k1='alpha'`. Then call it so that it prints:

```python
10 20 (30, 40) alpha beta {'debug': True}
```

Use `functools.partial`, not a wrapper function.

In [2]:
def report(a, b, *args, k1, k2, **kwargs):
    print(a, b, args, k1, k2, kwargs)

# Solution
p = partial(report, 10, k1='alpha')
p(20, 30, 40, k2='beta', debug=True)

10 20 (30, 40) alpha beta {'debug': True}


### Explanation

`partial(report, 10, k1='alpha')` freezes the first positional argument and one keyword-only argument. The remaining arguments are supplied when the partial object is called.

## Problem 2: Keyword Overriding

Create a partial function from `power(base, exponent)` that normally cubes a number. Then demonstrate that the exponent can still be overridden using a keyword argument.

In [3]:
def power(base, exponent):
    return base ** exponent

# Solution
cube = partial(power, exponent=3)

print(cube(2))              # 2 ** 3
print(cube(2, exponent=4))  # 2 ** 4

8
16


### Explanation

Keywords supplied later override keywords stored in the partial object. However, passing another positional argument for `exponent` would cause a conflict.

## Problem 3: Diagnose a TypeError

Explain why the following code raises a `TypeError`, then fix it.

```python
cube = partial(power, exponent=3)
cube(2, 4)
```

In [4]:
# Solution
try:
    cube(2, 4)
except TypeError as ex:
    print(type(ex).__name__, ex)

# Correct approaches
print(cube(2))
print(cube(2, exponent=4))

TypeError power() got multiple values for argument 'exponent'
8
16


### Explanation

`cube(2, 4)` passes `4` positionally as `exponent`, but `exponent=3` was already stored in the partial. Python receives two values for the same parameter.

## Problem 4: Sort Points by Distance from a Configurable Anchor

Write a distance-squared function that accepts two points. Then use `partial` to sort a list of points by distance from `(3, 4)`.

In [5]:
points = [(0, 0), (3, 5), (10, 10), (3, 4), (1, 1)]

def dist2(anchor, point):
    return (anchor[0] - point[0]) ** 2 + (anchor[1] - point[1]) ** 2

# Solution
from_anchor = partial(dist2, (3, 4))
sorted_points = sorted(points, key=from_anchor)
sorted_points

[(3, 4), (3, 5), (1, 1), (0, 0), (10, 10)]

### Explanation

`sorted` needs a one-argument key function. `dist2` needs two arguments. `partial(dist2, (3, 4))` adapts the two-argument function into a one-argument function.

## Problem 5: Mutable Object Trap

Create a partial function that stores a list as a fixed argument. Mutate the list after creating the partial. Show that the partial sees the mutation. Then create a safer version.

In [6]:
def show_config(config, value):
    return config, value

# Mutable version
config = ['debug']
f = partial(show_config, config)

print(f(100))
config.append('verbose')
print(f(100))

# Safer version: freeze a tuple copy
safe_config = ['debug']
safe_f = partial(show_config, tuple(safe_config))
safe_config.append('verbose')
print(safe_f(100))

(['debug'], 100)
(['debug', 'verbose'], 100)
(('debug',), 100)


### Explanation

`partial` stores object references, not deep copies. If the stored object is mutable, later mutations are visible through the partial. Use immutable objects or explicit copies when needed.

## Problem 6: Partial as a Callback Adapter

Suppose a library calls callbacks with exactly one argument: `result`. You want your callback to also know the username and timestamp format. Use `partial` to adapt the callback.

In [7]:
def fake_async_operation(callback):
    result = {'status': 'ok', 'items': 3}
    return callback(result)

def audit_log(user, date_format, result):
    now = datetime.now().strftime(date_format)
    return f'[{now}] user={user}, result={result}'

# Solution
callback = partial(audit_log, 'admin', '%Y-%m-%d')
fake_async_operation(callback)

"[2026-05-15] user=admin, result={'status': 'ok', 'items': 3}"

### Explanation

The library expects `callback(result)`. The original function expects `audit_log(user, date_format, result)`. `partial` pre-fills the first two arguments.

## Problem 7: Inspect a Partial Object

Create a partial object and inspect its `.func`, `.args`, and `.keywords` attributes.

In [8]:
def connect(host, port, *, timeout=10, ssl=False):
    return {
        'host': host,
        'port': port,
        'timeout': timeout,
        'ssl': ssl
    }

# Solution
secure_localhost = partial(connect, 'localhost', ssl=True)

print(secure_localhost.func)
print(secure_localhost.args)
print(secure_localhost.keywords)
print(secure_localhost(443, timeout=5))

<function connect at 0x000001E67C6C77E0>
('localhost',)
{'ssl': True}
{'host': 'localhost', 'port': 443, 'timeout': 5, 'ssl': True}


### Explanation

Partial objects are callable objects. They store the original function, frozen positional arguments, and frozen keyword arguments.

## Problem 8: Preserve Metadata with `update_wrapper`

`partial` objects do not automatically look like the original function when inspected. Create a partial and improve its metadata using `functools.update_wrapper`.

In [9]:
def format_price(currency, amount):
    """Format a numeric amount as a currency string."""
    return f'{currency}{amount:,.2f}'

# Solution
usd = partial(format_price, '$')
print(getattr(usd, '__name__', 'no __name__'))

update_wrapper(usd, format_price)
print(usd.__name__)
print(usd.__doc__)
print(usd(1234.5))

no __name__
format_price
Format a numeric amount as a currency string.
$1,234.50


### Explanation

`update_wrapper` can copy useful metadata such as `__name__` and `__doc__`. This is helpful when partials are used in frameworks, decorators, or debugging tools.

## Problem 9: Build Specialized Validators

Write a generic `between` validator and use `partial` to create `is_percentage`, which checks whether a number is between `0` and `100`, inclusive.

In [10]:
def between(value, *, low, high, inclusive=True):
    if inclusive:
        return low <= value <= high
    return low < value < high

# Solution
is_percentage = partial(between, low=0, high=100)

tests = [-1, 0, 50, 100, 101]
[(x, is_percentage(x)) for x in tests]

[(-1, False), (0, True), (50, True), (100, True), (101, False)]

### Explanation

The partial fixes the validator configuration while leaving the actual value open.

## Problem 10: Partial with `map`

Use `partial` with `operator.mul` to double every number in a list.

In [11]:
numbers = [1, 2, 3, 4, 5]

# Solution
double = partial(mul, 2)
list(map(double, numbers))

[2, 4, 6, 8, 10]

### Explanation

`mul` takes two arguments. `partial(mul, 2)` creates a one-argument function equivalent to `lambda x: 2 * x`.

## Problem 11: Partial for Reusable Sort Keys

Given a list of dictionaries, create reusable sort key functions for sorting by `name`, `age`, and `score`.

In [12]:
users = [
    {'name': 'Ada', 'age': 36, 'score': 98},
    {'name': 'Grace', 'age': 32, 'score': 95},
    {'name': 'Linus', 'age': 28, 'score': 91},
]

# Solution
key_by_name = partial(itemgetter, 'name')()
key_by_age = partial(itemgetter, 'age')()
key_by_score = partial(itemgetter, 'score')()

print(sorted(users, key=key_by_name))
print(sorted(users, key=key_by_age))
print(sorted(users, key=key_by_score, reverse=True))

[{'name': 'Ada', 'age': 36, 'score': 98}, {'name': 'Grace', 'age': 32, 'score': 95}, {'name': 'Linus', 'age': 28, 'score': 91}]
[{'name': 'Linus', 'age': 28, 'score': 91}, {'name': 'Grace', 'age': 32, 'score': 95}, {'name': 'Ada', 'age': 36, 'score': 98}]
[{'name': 'Ada', 'age': 36, 'score': 98}, {'name': 'Grace', 'age': 32, 'score': 95}, {'name': 'Linus', 'age': 28, 'score': 91}]


### Explanation

`itemgetter('name')` already returns a callable. Here, `partial(itemgetter, 'name')()` demonstrates that partial can also configure callable factories. In practice, `itemgetter('name')` is simpler.

## Problem 12: Avoid Late Binding in Loops

Create a list of multiplier functions for factors `1`, `2`, `3`, and `4`. Use `partial` to avoid the common late-binding closure bug.

In [13]:
# Problematic closure version
bad_funcs = []
for factor in range(1, 5):
    bad_funcs.append(lambda x: factor * x)

print([f(10) for f in bad_funcs])

# Solution using partial
good_funcs = []
for factor in range(1, 5):
    good_funcs.append(partial(mul, factor))

print([f(10) for f in good_funcs])

[40, 40, 40, 40]
[10, 20, 30, 40]


### Explanation

The lambda version closes over the variable `factor`, not its value at each iteration. The partial version stores the current factor immediately.

## Problem 13: Partial Methods vs Partial Functions

Use `partial` to create a function that calls `str.replace` to remove hyphens from strings.

In [14]:
# Solution
remove_hyphens = partial(str.replace, old='-', new='')

print(remove_hyphens('123-45-6789'))
print(remove_hyphens('a-b-c'))

TypeError: replace() takes at least 2 positional arguments (0 given)

### Explanation

`str.replace` is an unbound method-like descriptor. The string instance is supplied later as the first argument.

## Problem 14: Build a Small Notification System

You have a generic `notify` function. Use partial functions to create three specialized notification functions:

- `notify_admin`
- `notify_devteam`
- `notify_security`

Each should require only the message body at call time.

In [15]:
def notify(to, subject, body, *, priority='normal'):
    return {
        'to': to,
        'subject': subject,
        'body': body,
        'priority': priority
    }

# Solution
notify_admin = partial(notify, 'admin@example.com', 'Admin Notice')
notify_devteam = partial(notify, 'dev@example.com', 'Dev Notice')
notify_security = partial(
    notify,
    'security@example.com',
    'Security Alert',
    priority='high'
)

print(notify_admin('Daily report ready'))
print(notify_devteam('Build completed'))
print(notify_security('Suspicious login detected'))

{'to': 'admin@example.com', 'subject': 'Admin Notice', 'body': 'Daily report ready', 'priority': 'normal'}
{'to': 'dev@example.com', 'subject': 'Dev Notice', 'body': 'Build completed', 'priority': 'normal'}
{'to': 'security@example.com', 'subject': 'Security Alert', 'body': 'Suspicious login detected', 'priority': 'high'}


### Explanation

Partial functions are useful when a general-purpose function needs to be specialized for common cases.

## Problem 15: Signature Reasoning

Use `inspect.signature` to compare the signature of a function and the apparent signature of a partial object.

In [16]:
def api_call(method, endpoint, *, timeout=30, retries=3):
    return method, endpoint, timeout, retries

# Solution
get_user = partial(api_call, 'GET', '/users', timeout=10)

print(inspect.signature(api_call))
print(inspect.signature(get_user))
print(get_user(retries=5))

(method, endpoint, *, timeout=30, retries=3)
(*, timeout=10, retries=3)
('GET', '/users', 10, 5)


### Explanation

`inspect.signature` understands partial objects and shows which parameters remain available after partial application.

## Problem 16: Design Challenge — Configurable Pipeline Step

Create a reusable text-cleaning pipeline. Write a generic `replace_text` function and use partials to create steps that remove commas, periods, and extra spaces.

In [17]:
def replace_text(text, old, new):
    return text.replace(old, new)

# Solution
remove_commas = partial(replace_text, old=',', new='')
remove_periods = partial(replace_text, old='.', new='')
collapse_double_spaces = partial(replace_text, old='  ', new=' ')

def apply_pipeline(text, steps):
    for step in steps:
        text = step(text)
    return text

text = 'Hello,  world. This,  is Python.'
steps = [remove_commas, remove_periods, collapse_double_spaces]

apply_pipeline(text, steps)

'Hello world This is Python'

### Explanation

Each partial has the same one-argument interface: it accepts text and returns transformed text. This makes them easy to compose in a pipeline.

## Problem 17: Partial vs Lambda Refactoring

Refactor the following lambda into a partial:

```python
key = lambda user: score_user(user, weights={'activity': 2, 'reputation': 5})
```

In [18]:
def score_user(user, *, weights):
    return user['activity'] * weights['activity'] + user['reputation'] * weights['reputation']

users = [
    {'name': 'Ada', 'activity': 10, 'reputation': 4},
    {'name': 'Grace', 'activity': 5, 'reputation': 10},
    {'name': 'Linus', 'activity': 7, 'reputation': 7},
]

# Solution
key = partial(score_user, weights={'activity': 2, 'reputation': 5})
sorted(users, key=key, reverse=True)

[{'name': 'Grace', 'activity': 5, 'reputation': 10},
 {'name': 'Linus', 'activity': 7, 'reputation': 7},
 {'name': 'Ada', 'activity': 10, 'reputation': 4}]

### Explanation

The partial stores the scoring configuration and leaves the `user` argument open.

## Problem 18: Advanced Debugging — Shared Keyword Dictionary

Investigate whether mutating a dictionary after passing it as a keyword argument to `partial` affects future calls.

In [19]:
def render(template, *, context):
    return template.format(**context)

# Solution
context = {'name': 'Ada'}
greet = partial(render, 'Hello, {name}!', context=context)

print(greet())

context['name'] = 'Grace'
print(greet())

# Safer version
safe_greet = partial(render, 'Hello, {name}!', context=dict(context))
context['name'] = 'Linus'
print(safe_greet())

Hello, Ada!
Hello, Grace!
Hello, Grace!


### Explanation

Keyword arguments stored in a partial also store object references. A mutable dictionary can still be changed externally unless copied.

## Problem 19: Create a Function Factory with Partial

Write a factory function `make_formatter(prefix, suffix)` that returns a partial function. The returned function should accept only a value.

In [20]:
def surround(value, *, prefix='', suffix=''):
    return f'{prefix}{value}{suffix}'

# Solution
def make_formatter(prefix, suffix):
    return partial(surround, prefix=prefix, suffix=suffix)

as_tag = make_formatter('<strong>', '</strong>')
as_quote = make_formatter('"', '"')

print(as_tag('Important'))
print(as_quote('Python'))

<strong>Important</strong>
"Python"


### Explanation

`partial` is often useful inside factory functions because it can return a configured callable without manually defining an inner function.

## Problem 20: Best-Practice Review

For each statement, decide whether it is generally a good practice.

1. Use `partial` when adapting a many-argument function to an API that expects fewer arguments.
2. Use `partial` even when it makes the code harder to read.
3. Be cautious when freezing mutable objects.
4. Use `partial` to avoid repeated configuration arguments.
5. Assume that `partial` deep-copies its arguments.

In [21]:
# Solution
answers = {
    1: True,
    2: False,
    3: True,
    4: True,
    5: False,
}

answers

{1: True, 2: False, 3: True, 4: True, 5: False}

### Explanation

`partial` is best when it improves clarity, reduces repetition, or adapts a function to a required interface. It should not be used just to appear clever, and it does not protect you from mutable-state issues.